# 05 · Extension 1: the implicit question

The same sentences of dataset **v2**, but instead of *"Is this person being sarcastic?"* the model is asked
**"Is this person actually happy about it?"**. Answering correctly needs the sarcasm to be understood: the correct
answer follows the **situation**, not the words of the phrase.

| | situation | phrase | correct | literal reading of the phrase |
|---|---|---|---|---|
| P | pleasant | positive | **Yes** | Yes |
| S | unpleasant | positive | **No** | Yes ✗ |
| N | unpleasant | negative | **No** | No |
| Q | pleasant | negative | (ambiguous) | No |

**S is the key condition**: a system that reads the phrase literally fails exactly there.

Three systems:
1. **Qwen (base)**: the full model with the implicit question.
2. **ACDC circuit** (8147 edges, found with the *explicit* question): answers alone (mean ablation, mean computed on
   the implicit prompts). If it reproduces the model, the mechanism is not tied to the word "sarcastic".
3. **Custom mechanism**: the separation into situation and phrase is **given** by the dataset, SST-2 classifies the
   two parts, the rule decides "sarcastic = the two disagree", and the answer is "happy = positive phrase, inverted
   if sarcastic". With a given separation this reduces to the sentiment of the situation (noted in the report).
   For reference also the **literal** reader: "happy = positive phrase".

Causal test (H3): in the full model, remove the sarcasm direction of 03a (layers 19–21) or the layer-18 writers
(m18 + 5 key heads), against random controls, and see whether the accuracy on S drops.

**Runtime:** L4 ≈ 30–45 min. Needs from Drive: the circuit of 02d, the direction of 03a and (optionally) the
per-prompt results of 03d (for the consistency with the explicit question).

## Setup

In [ ]:
# Get (or update) the code from GitHub
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q -U accelerate bitsandbytes

In [ ]:
import os, json, time, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src.model import load_model, model_info, DEV_MODEL, MAIN_MODEL
from src.prompts import load_dataset, encode, chat_text, data_tag
from src.eap import build_graph
from src.standalone import MeanReference, StandaloneEvaluator, standalone_d
from src.baseline import sst2_p, compare
from src.pathpatch import mean_last, ablate_d
from src.implicit import IMPLICIT_Q, implicit_prompt, project_out_d, truth_implicit, summary_implicit
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
MODEL = MAIN_MODEL          # DEV_MODEL (3B) for quick tests
DATA = "v2"
QUESTION = IMPLICIT_Q       # "Is this person actually happy about it?"
CIRCUIT = "v3b/acdc_mask_tau0.0.npy"     # the final circuit (02d)
DIRECTION = "sarcasm_direction_L19.npy"  # from 03a
PROJECT_LAYERS = [19, 20, 21]            # where the sarcasm direction is removed
KEY_LAYER, KEY_HEADS = 18, [15, 18, 4, 13, 16]
N_RANDOM = 3                             # random controls for the causal tests
# ------------------------------------------
TAG_MODEL = MODEL.split("/")[-1]
TAG = TAG_MODEL + data_tag(DATA)
SUB = f"05_implicit/{TAG}"
mount_drive()
print("results go to:", results_dir(SUB))
set_seed(0)

## Data: P, S, N, Q with the implicit question

In [ ]:
df = load_dataset(DATA).sort_values(["split", "triple_id", "condition"]).reset_index(drop=True)
df["group"] = [s if s == "discovery" else f"heldout ({t})" for s, t in zip(df.split, df.heldout_type)]
GROUPS = ["discovery", "heldout (phrase)", "heldout (frame)"]

# Q: P's sentence with N's phrase (as in 03a / 03d)
q_rows = []
for tid, g in df.groupby("triple_id"):
    r = g.set_index("condition")
    P, N = r.loc["P"], r.loc["N"]
    q_rows.append(dict(triple_id=tid, condition="Q", text=P.text[: -len(P.frame)] + N.frame, situation=P.situation,
                       situation_type=P.situation_type, frame=N.frame, frame_type=N.frame_type, group=P.group,
                       split=P.split, heldout_type=P.heldout_type,
                       prompt=P.prompt.replace(P.text, P.text[: -len(P.frame)] + N.frame)))
cols = list(q_rows[0].keys())
D4 = pd.concat([df[cols], pd.DataFrame(q_rows)], ignore_index=True)
D4["prompt_explicit"] = D4.prompt
D4["prompt"] = [implicit_prompt(p, QUESTION) for p in D4.prompt_explicit]
D4["truth"] = truth_implicit(D4)            # True = "Yes, happy" (the situation is pleasant)
PSN = (D4.condition != "Q").values
print({g: int((D4.group == g).sum()) for g in GROUPS}, "prompts (P, S, N, Q)")
print("example (S):", D4[D4.condition == "S"].prompt.iloc[0])

## Custom mechanism (given separation) and the literal reader

The situation and the phrase are taken from the dataset columns. SST-2 gives p(pleasant) for the situation and
p(positive) for the phrase.

In [ ]:
SIT = D4.situation.drop_duplicates().tolist()
PHR = D4.frame.drop_duplicates().tolist()
P_CLS = cached("sst2_p", lambda: {"situation": dict(zip(SIT, sst2_p(SIT))), "phrase": dict(zip(PHR, sst2_p(PHR)))},
               sub=SUB)
torch.cuda.empty_cache()
p_sit = D4.situation.map(P_CLS["situation"]).values
p_phr = D4.frame.map(P_CLS["phrase"]).values
pleasant, positive = p_sit > 0.5, p_phr > 0.5
sarcastic = positive != pleasant                  # the disagree rule
D4["yes custom"] = positive ^ sarcastic           # happy = phrase, inverted if sarcastic (= pleasant situation)
D4["yes literal"] = positive                      # happy = the words of the phrase
D4["custom sarcastic"] = sarcastic
assert (D4["yes custom"] == pleasant).all()
sit_ok = ((p_sit > 0.5) == (D4.situation_type == "pleasant")).values
phr_ok = ((p_phr > 0.5) == (D4.frame_type == "positive")).values
print(f"SST-2 accuracy on the parts (unique items): situations "
      f"{np.mean([(P_CLS['situation'][s] > 0.5) == (t == 'pleasant') for s, t in D4[['situation', 'situation_type']].drop_duplicates().values]):.3f}, "
      f"phrases {np.mean([(P_CLS['phrase'][f] > 0.5) == (t == 'positive') for f, t in D4[['frame', 'frame_type']].drop_duplicates().values]):.3f}")

## Qwen and the circuit on every prompt

In [ ]:
t0 = time.time()
model, tok = load_model(MODEL)
G = build_graph(model)
print(MODEL, model_info(model), f"loaded in {(time.time() - t0) / 60:.1f} min")
path = os.path.join(results_dir(""), f"02_acdc/{TAG}/{CIRCUIT}")
assert os.path.exists(path), f"circuit not found: {path}"
MASK = np.load(path)
print(f"circuit: {int(MASK.sum())} edges")

encs = [encode(tok, r) for r in D4.itertuples()]
IDS = [e.ids for e in encs]
IDS_PSN = [x for x, m in zip(IDS, PSN) if m]
PREFIX = int(np.median([e.regions["situation"][0] for e in encs]))
SUFFIX = int(np.median([len(e.ids) - (e.regions["phrase"][-1] + 1) for e in encs]))
print("question tokens (first prompt):", [tok.decode([encs[0].ids[p]]) for p in encs[0].regions["question"]])

t0 = time.time()
REF = MeanReference(model, IDS_PSN, G, prefix=PREFIX, suffix=SUFFIX)
print("mean reference (implicit prompts):", REF.counts, f"| {time.time() - t0:.0f} s")

# sanity: all edges = full model
T0 = pd.Series([len(x) for x in IDS_PSN]).mode()[0]
part = [x for x in IDS_PSN if len(x) == T0][:16]
ev = StandaloneEvaluator(model, tok, part, REF, G)
gap = np.abs(ev.d(G.valid) - ev.d_clean).max()
print(f"all edges: max |d - d_full| = {gap:.5f}")
assert gap < 1e-3, "sanity check failed: send the output to Claude"
del ev; torch.cuda.empty_cache()
print("PASS")

In [ ]:
t0 = time.time()
DD = cached("d_full_and_circuit", lambda: standalone_d(model, tok, IDS, {"circuit": MASK}, REF, G), sub=SUB)
print(f"{(time.time() - t0) / 60:.1f} min")
D4["d Qwen"], D4["d circuit"] = DD["full"], DD["circuit"]
D4["yes Qwen"], D4["yes circuit"] = D4["d Qwen"] > 0, D4["d circuit"] > 0

## 1 · Main table

Per group, on P, S, N: accuracy per condition (**acc S** is the key number), balanced accuracy, triple accuracy
(P = Yes, S = No, N = No), precision and recall of "No" (not happy), "literal on S" (share of S answered "Yes",
i.e. following the words), and the share of "Yes" on Q (no ground truth).

In [ ]:
SYSTEMS = ["Qwen", "circuit", "custom", "literal"]
MAIN = pd.concat({g: pd.DataFrame({k: summary_implicit(D4[D4.group == g][f"yes {k}"], D4[D4.group == g])
                                   for k in SYSTEMS}).T for g in GROUPS})
MAIN.index.names = ["group", "system"]
display(MAIN.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, g in zip(axes, GROUPS):
    x = np.arange(3)
    for j, k in enumerate(SYSTEMS):
        ax.bar(x + (j - 1.5) * 0.2, MAIN.loc[g].loc[k, ["acc P", "acc S", "acc N"]], 0.2, label=k)
    ax.set_xticks(x); ax.set_xticklabels(["P (Yes)", "S (No)", "N (No)"]); ax.set_title(g); ax.set_ylim(0, 1.05)
axes[0].set_ylabel("accuracy"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "accuracy_per_condition.png"), dpi=150); plt.show()

## 2 · Consistency with the explicit question

For the S sentences: does the model answer the implicit question correctly when it detects the sarcasm (explicit
question "Yes"), and when it does not? Uses the per-prompt results of 03d if they exist, otherwise computes the
explicit d.

In [ ]:
p3d = os.path.join(results_dir(""), f"03d_baseline/{TAG}/per_prompt.csv")
if os.path.exists(p3d):
    E = pd.read_csv(p3d)[["triple_id", "condition", "d Qwen"]].rename(columns={"d Qwen": "d explicit"})
    D4 = D4.drop(columns=["d explicit"], errors="ignore").merge(E, on=["triple_id", "condition"], how="left")
    print("explicit d from 03d")
else:
    from src.metric import d_batch
    ids_exp = [tok(chat_text(tok, p), add_special_tokens=False)["input_ids"] for p in D4.prompt_explicit]
    D4["d explicit"] = cached("d_explicit", lambda: d_batch(model, tok, ids_exp), sub=SUB)
    print("explicit d computed")
D4["yes explicit"] = D4["d explicit"] > 0

rows = []
for g in GROUPS:
    s = D4[(D4.group == g) & (D4.condition == "S")]
    det = s["yes explicit"].values
    ok = ~s["yes Qwen"].values                   # correct implicit answer on S = "No"
    rows.append({"group": g, "S detected (explicit Yes)": det.mean(),
                 "implicit correct | detected": ok[det].mean() if det.any() else np.nan,
                 "implicit correct | not detected": ok[~det].mean() if (~det).any() else np.nan,
                 "implicit correct (all S)": ok.mean()})
CONS = pd.DataFrame(rows).set_index("group")
display(CONS.round(3))

## 3 · Agreement and shared errors (P, S, N)

As in 03d: agreement and Cohen's κ with Qwen, and how many of Qwen's errors each system also makes (against the
number expected if the errors were independent).

In [ ]:
rows = []
for g in GROUPS:
    s = D4[(D4.group == g) & (D4.condition != "Q")]
    for b in SYSTEMS[1:]:
        c = compare(s[f"yes {b}"], s["yes Qwen"], s.truth)
        rows.append(dict(group=g, system=b, agreement=c["agreement"], kappa=c["kappa"],
                         **{k: c[k] for k in ["errors (system)", "errors (reference)", "shared errors", "shared if independent"]}))
AGR = pd.DataFrame(rows).rename(columns={"errors (reference)": "errors (Qwen)"}).set_index(["group", "system"])
display(AGR.round(3))

## 4 · Causal test: does the model use the sarcasm mechanism for the implicit question?

On the full model with the implicit question:
* **direction**: remove the sarcasm direction of 03a from the last token at layers 19–21 (vs. random directions);
* **writers**: replace m18 and the 5 key heads of layer 18 by their mean at the last token (vs. 5 random heads).

As a check, the same ablations on the **explicit** question should lower the detection of sarcasm.
If the accuracy on S drops much more than with the random controls, the implicit answer uses the mechanism.

In [ ]:
pdir = os.path.join(results_dir(""), f"03a_direction/{TAG}/{DIRECTION}")
assert os.path.exists(pdir), f"direction not found: {pdir}"
U = np.load(pdir).astype(np.float32)
rng = np.random.default_rng(0)
RAND_DIRS = [rng.standard_normal(U.shape).astype(np.float32) for _ in range(N_RANDOM)]
H = model_info(model)["n_heads"]
RAND_HEADS = [sorted(rng.choice([h for h in range(H) if h not in KEY_HEADS], 5, replace=False).tolist())
              for _ in range(N_RANDOM)]
IDS_EXP = [tok(chat_text(tok, p), add_special_tokens=False)["input_ids"] for p in D4.prompt_explicit]

def ablations(ids):
    z, m = mean_last(model, [x for x, k in zip(ids, PSN) if k], KEY_LAYER)
    out = {"none": None,
           "direction (L19-21)": project_out_d(model, tok, ids, PROJECT_LAYERS, U),
           "writers (m18 + 5 heads)": ablate_d(model, tok, ids, KEY_LAYER, KEY_HEADS, True, z, m)}
    for i, v in enumerate(RAND_DIRS):
        out[f"random direction #{i}"] = project_out_d(model, tok, ids, PROJECT_LAYERS, v)
    for i, hs in enumerate(RAND_HEADS):
        out[f"random 5 heads #{i}"] = ablate_d(model, tok, ids, KEY_LAYER, hs, False, z, m)
    return out

t0 = time.time()
ABL = cached("ablations_implicit", lambda: ablations(IDS), sub=SUB)
ABL_EXP = cached("ablations_explicit", lambda: ablations(IDS_EXP), sub=SUB)
print(f"{(time.time() - t0) / 60:.1f} min")
ABL["none"], ABL_EXP["none"] = D4["d Qwen"].values, D4["d explicit"].values

rows = []
for g in GROUPS:
    m = (D4.group == g).values
    sub = D4[m]
    for k in ABL:
        si = summary_implicit(ABL[k][m] > 0, sub)
        se = sub.condition.values == "S"
        rows.append({"group": g, "ablation": k, "implicit: acc S": si["acc S"], "implicit: triple acc": si["triple acc"],
                     "implicit: bal. acc": si["bal. acc"],
                     "explicit: Yes on S": float((ABL_EXP[k][m][se] > 0).mean()),
                     "explicit: Yes on P, N": float((ABL_EXP[k][m][np.isin(sub.condition.values, ["P", "N"])] > 0).mean())})
CAUS = pd.DataFrame(rows).set_index(["group", "ablation"])
display(CAUS.round(3))

fig, ax = plt.subplots(figsize=(11, 3.6))
keys = list(ABL)
x = np.arange(len(keys))
for j, g in enumerate(GROUPS):
    ax.bar(x + (j - 1) * 0.27, CAUS.loc[g].loc[keys, "implicit: acc S"], 0.27, label=g)
ax.set_xticks(x); ax.set_xticklabels(keys, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("accuracy on S (implicit)"); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "causal_test.png"), dpi=150); plt.show()

## 5 · Your own sentences

In [ ]:
EXAMPLES = [
    ("my car breaking down on the highway", "What a fantastic way to start the week."),
    ("a relaxing weekend at the lake", "Ugh, terrible."),
    ("getting a promotion at work", "Just perfect."),
]
FRAME = "Today was all about {s}. {f}"
template = df.prompt.iloc[0].replace(df.text.iloc[0], "{}")
for s, f in EXAMPLES:
    text = FRAME.format(s=s, f=f)
    ids = tok(chat_text(tok, implicit_prompt(template.format(text), QUESTION)), add_special_tokens=False)["input_ids"]
    ev = StandaloneEvaluator(model, tok, [ids], REF, G)
    dq, dc = float(ev.d_clean[0]), float(ev.d(MASK)[0])
    del ev
    ps, pp = sst2_p([s])[0], sst2_p([f])[0]
    print(f"\n{text}\n  Q: {QUESTION}")
    print(f"  Qwen     d = {dq:+6.2f} -> {'Yes' if dq > 0 else 'No'}")
    print(f"  circuit  d = {dc:+6.2f} -> {'Yes' if dc > 0 else 'No'}")
    print(f"  custom   situation p(pleasant) = {ps:.2f}, phrase p(positive) = {pp:.2f}, sarcastic = {(pp > 0.5) != (ps > 0.5)}"
          f" -> {'Yes' if (pp > 0.5) ^ ((pp > 0.5) != (ps > 0.5)) else 'No'}   (literal: {'Yes' if pp > 0.5 else 'No'})")
torch.cuda.empty_cache()

## Save

In [ ]:
d = results_dir(SUB)
D4.drop(columns=["prompt", "prompt_explicit"]).to_csv(os.path.join(d, "per_prompt.csv"), index=False)
for name, t in [("main", MAIN), ("consistency", CONS), ("agreement", AGR), ("causal", CAUS)]:
    t.to_csv(os.path.join(d, f"{name}.csv"))
print("saved to", d)